In [1]:
#os stands for operating system and helps check the file system running code
import os

# Target file needed for data_prep.py
raw_file = "data/raw/variant_summary.txt"

# Only download and extract if the uncompressed file doesn't exist on this runtime
if not os.path.exists(raw_file):
    print("New Colab session detected. Downloading ClinVar data...")
    #! runs the code in terminal, mk dir makes a directory(folder), and -p finds the parent files
    !mkdir -p data/raw
    #downloads the file from the source
    !wget https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz -P data/raw/
    #gunzip takes the file and extracts it to the "raw" folder in "data", and f forces the code without popups
    !gunzip -f data/raw/variant_summary.txt.gz
    print("ClinVar download and extraction complete!")
else:
    print("ClinVar data is already present in this session runtime. Ready to go!")

New Colab session detected. Downloading ClinVar data...
--2026-08-06 00:38:13--  https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.12, 130.14.250.13, 2607:f220:41e:250::7, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 441792560 (421M) [application/x-gzip]
Saving to: ‘data/raw/variant_summary.txt.gz’

variant_summary.txt 100%[===================>] 421.33M  64.6MB/s    in 7.1s    

2026-08-06 00:38:20 (59.7 MB/s) - ‘data/raw/variant_summary.txt.gz’ saved [441792560/441792560]

ClinVar download and extraction complete!


In [3]:
import torch

print("GPU Active:", torch.cuda.get_device_name(0))


GPU Active: Tesla T4


In [1]:
import sys
import os
import pandas as pd
from pathlib import Path

# Find the project root
def find_project_root(marker="src/data_prep.py"):

    # Calls the class working directory from pathlib, and captures the starting location
    here = Path.cwd()

    # Creates an upward search path, checking one folder layer at a time until
    # candidate (the project root) is returned 
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
        
    # Incase the file is not found
    raise FileNotFoundError(
        print("Could not find the file")
    )

# Establishing the project paths and root
project_root = find_project_root()
src_path = project_root / "src"
data_path = project_root / "data"

print(f"Project root confirmed at: {project_root}")

# Insert src directory at top priority
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Import the processing function
from data_prep import build_full_dataset

# Defining the ClinVar file path (includes lots of variants, which are filtered in data_prep)
ClinVar_path = str(data_path / "raw" / "variant_summary.txt.gz")

# Unlike ClinVar, gnomAD includes seperated data for each gene. This takes
# the paths for each of the three genes.
gnomAD_csv_paths = {
    "MYH7": str(data_path / "raw" / "gnomAD_MYH7.csv"),
    "MYBPC3": str(data_path / "raw" / "gnomAD_MYBPC3.csv"),
    "TTN": str(data_path / "raw" / "gnomAD_TTN.csv")
}
out_path = str(data_path / "processed" / "dataset.csv")

# Runs data preparation pipeline and calls build_full_dataset from data_prep
print("Starting data preparation pipeline")
final_df = build_full_dataset(
    ClinVar_path=ClinVar_path,
    gnomAD_csv_paths=gnomAD_csv_paths,
    out_path=out_path
)

# Configure Pandas display options and show results
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

print("\n--- ENTIRE FINISHED DATASET ---")
display(final_df)

Project root confirmed at: c:\Users\jaira\Desktop\Official cardiac project\Cardiac-project
Starting data preparation pipeline


ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))